# E3 · Fidelidad de citas y disciplina del crítico (P3) → §9.3
# E4 · Coste, latencia y reproducibilidad (P4) → §9.4

Versión acotada (experimentos.md §15): un solo notebook para las dos preguntas que dependen de las
ejecuciones con API. No lanza nada: lee las tandas que ya ha ejecutado `ejecutar_lote.py`.

| Parte | Pregunta | Datos |
|---|---|---|
| **E3** | ¿Toda cifra del texto sale del informe, sin alterarse y sin citar métricas cerradas? ¿El crítico respeta el arbitraje? | Tanda **T100** (100 ejecuciones con lectura visual) |
| **E4b** | ¿Cuánto cuesta y cuánto tarda un análisis? ¿Cuántos fallan? | T100, incluidos los intentos fallidos |
| **E4c** | ¿Es reproducible? | Tanda **R**: 5 imágenes × 3 repeticiones |

**Instrumentos.** El verificador (`experimentos/verificador.py`) extrae cada cita `[campo = valor]`, la resuelve
contra el `informe` del propio `salida_*.json` y la clasifica: `exacta` (redondeo correcto a los decimales
escritos), `truncada` (coincide con cortar dígitos, no con redondear), `alterada`, `texto_distinto`,
`null_citado`, `no_numerica` o `campo_inexistente`. Aparte comprueba que el informe transcrito por el LLM
coincide con el de la herramienta (caché de gates, ejecutada con el sistema congelado) y recomputa el
arbitraje con `PrioridadEngine`.

Criterios (fijados en `config_experimentos.yaml` antes de ejecutar):

- **E3:** 0 citas a métricas cerradas; ≥ 95 % de citas exactas; 100 % de críticas con `afirmaciones` en el orden
  del arbitraje.
- **E4:** 100 % de informes idénticos (transcripción = herramienta) y arbitraje idéntico entre las 3 repeticiones de R.

## 1. Configuración, criterios y congelación

In [ ]:
import sys, json, re, warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ / "experimentos") not in sys.path:
    sys.path.insert(0, str(RAIZ / "experimentos"))
import comun
comun.preparar_sys_path()
import verificador as V

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

_warn = warnings.warn   # CrewAI sustituye warnings.warn por una versión incompatible con matplotlib
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    from tfg_multiagente_fotografia.tools import prioridad_tool as pt   # <- EL CÓDIGO REAL del arbitraje
warnings.warn = _warn

CFG = comun.cargar_config()
CRIT_E3, CRIT_E4 = CFG["E3"]["criterios"], CFG["E4"]["criterios"]
PRECIOS = CFG["precios"]
TANDAS = comun.ruta_config("tandas")
RES3, RES4 = comun.dir_resultados("E3"), comun.dir_resultados("E4")

AZUL, NARANJA, AQUA, TINTA, TINTA2, GRIS, REJILLA, EJE = ("#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e",
                                                          "#898781", "#e1e0d9", "#c3c2b7")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11,
    "axes.titlecolor": TINTA, "axes.labelcolor": TINTA2, "axes.edgecolor": EJE,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": REJILLA, "grid.linewidth": 0.6, "xtick.color": GRIS, "ytick.color": GRIS,
    "lines.linewidth": 2, "legend.frameon": False,
})
pd.set_option("display.max_colwidth", 90)

PROC = comun.procedencia()
CONGELADO = comun.comprobar_congelacion()
CONGELACION = json.loads((RAIZ / "experimentos" / "congelacion.json").read_text(encoding="utf-8"))
print(f"commit {PROC['commit'][:7]} · congelado: {CONGELADO} (congelación del commit {CONGELACION['commit'][:7]})")
print("Criterios E3:", CRIT_E3, "· E4:", CRIT_E4)

## 2. Carga de las tandas

Cada ejecución tiene su `registro.json` (lo escribe el lanzador). Solo cuentan como resultado las ejecuciones hechas
**con el sistema congelado en el mismo commit**; el notebook lo comprueba. Los intentos fallidos están archivados en
`{tanda}/fallidos/` y cuentan para la tasa de fallos y para el gasto.

In [ ]:
def registros(tanda):
    filas = []
    for r in sorted((TANDAS / tanda).glob("**/registro.json")):
        reg = json.loads(r.read_text(encoding="utf-8"))
        reg["dir"] = r.parent
        reg["archivado"] = "fallidos" in r.parts
        filas.append(reg)
    return filas


REG = {t: registros(t) for t in ("T100", "R")}
for t, regs in REG.items():
    assert regs, f"No hay ejecuciones de la tanda {t}: lánzala con ejecutar_lote.py"
    commits = {r["sistema"]["commit"] for r in regs}
    assert commits == {CONGELACION["commit"]}, f"{t}: ejecuciones con otro commit {commits}"
    assert all(r["sistema"]["congelado"] for r in regs), f"{t}: hay ejecuciones sin congelar"

T100 = [r for r in REG["T100"] if not r["archivado"]]
R = [r for r in REG["R"] if not r["archivado"]]
STEMS_T100 = sorted({r["imagen"] for r in T100}, key=lambda s: int(s[6:]))
assert len(STEMS_T100) == 100 and all(r["estado"] == "ok" for r in T100), "T100 incompleta: reanuda el lanzador"
LISTA_R = CFG["listas"]["R"]
assert sorted((r["imagen"], r["rep"]) for r in R) == sorted((s, k) for s in LISTA_R for k in (1, 2, 3)), "R incompleta"
assert all(r["estado"] == "ok" for r in R)
print(f"T100: {len(T100)} ejecuciones ok + {sum(r['archivado'] for r in REG['T100'])} intentos fallidos archivados")
print(f"R: {len(R)} ejecuciones ok ({len(LISTA_R)} imágenes × 3) + {sum(r['archivado'] for r in REG['R'])} fallidos")

G = comun.cargar_cache("gates")["imagenes"]   # informes de la HERRAMIENTA (engines, sistema congelado)

## 3. E3 · Verificación de las citas (T100)

Para cada ejecución: las citas de los 4 especialistas (`diagnostico`) y del crítico (afirmaciones, tensiones y su
resolución por W, salvedades y síntesis), los controles del crítico y la fidelidad de la transcripción del informe.

Una cita a una métrica **cerrada** es violación si cita su `valor`/`valor_norm` (o la métrica a secas) o un campo de
evidencia condicionado por ella (`verificador.CAMPOS_CONDICIONADOS`); citar `.confianza` no lo es. Las citas del crítico
a W, a la cobertura o a la etiqueta se resuelven contra el arbitraje; si no usan la forma canónica (`W(dim)`,
`vector_pesos.dim`, `cobertura_por_dimension.dim`) se marcan como `forma_canonica = False`, sin penalizar la cifra.

In [ ]:
def auditar_ejecucion(reg):
    d = reg["dir"]
    salidas = V.cargar_salidas(d)
    critica = json.loads((d / "salida_critica.json").read_text(encoding="utf-8"))
    orquestador = json.loads((d / "salida_orquestador.json").read_text(encoding="utf-8"))
    etiqueta = critica["arbitraje"]["etiqueta_contexto"]
    recomputado = pt.PrioridadEngine(raiz=d).arbitrar(str(RAIZ / "data" / f"{reg['imagen']}.jpg"),
                                                      etiqueta).model_dump(mode="json")
    base = {"tanda": reg["tanda"], "rep": reg["rep"], "imagen": reg["imagen"]}
    citas = []
    for dim, s in salidas.items():
        citas += [{**base, **f} for f in V.auditar_especialista(s, dim)]
    c, controles = V.auditar_critica(critica, salidas, recomputado)
    citas += [{**base, **f} for f in c]
    controles.update(base, etiqueta_critico=etiqueta,
                     etiqueta_orquestador=orquestador.get("etiqueta_contexto"),
                     etiqueta_igual_orquestador=etiqueta == orquestador.get("etiqueta_contexto"))
    fid = [{**base, "dimension": dim, **f}
           for dim, s in salidas.items() for f in V.fidelidad_transcripcion(s["informe"], G[reg["imagen"]]["informes"][dim])]
    return citas, controles, fid, critica["arbitraje"]


def auditar_tanda(regs):
    citas, controles, fid, arbs = [], [], [], {}
    for reg in regs:
        c, k, f, a = auditar_ejecucion(reg)
        citas += c; controles.append(k); fid += f; arbs[(reg["imagen"], reg["rep"])] = a
    return pd.DataFrame(citas), pd.DataFrame(controles), pd.DataFrame(fid), arbs


CITAS, CONTROLES, FIDELIDAD, _ = auditar_tanda(T100)
print(f"{len(CITAS)} citas en {len(CONTROLES)} ejecuciones · {len(FIDELIDAD)} campos de informe comparados")

### 3.1 Tabla de citas por autor

`% exactas` se calcula sobre **todas** las citas del autor (un campo inexistente cuenta como no exacta). `truncada`
separa el error típico de cortar dígitos en vez de redondear.

In [ ]:
ORDEN_AUTOR = ["composicion_espacial", "lineas_direccion", "espacio_aislamiento_sujeto", "luz_tono", "critico"]
CLASES = ["exacta", "truncada", "alterada", "texto_distinto", "null_citado", "no_numerica", "campo_inexistente"]


def tabla_autores(C):
    t = pd.crosstab(C.autor, C.clase).reindex(index=ORDEN_AUTOR, columns=CLASES, fill_value=0)
    t.loc["total"] = t.sum()
    n = t.sum(axis=1)
    out = pd.DataFrame({"citas": n, "citas por ejecución": (n / C.groupby(["imagen", "rep"]).ngroups).round(1)})
    out["% campo existente"] = 100 * (n - t["campo_inexistente"]) / n
    for c in ("exacta", "truncada", "alterada"):
        out[f"% {c}s"] = 100 * t[c] / n
    out["otras"] = t[["texto_distinto", "null_citado", "no_numerica"]].sum(axis=1)
    a_cerr = C.groupby("autor").a_cerrada.sum().reindex(ORDEN_AUTOR, fill_value=0)
    out["citas a cerradas"] = [*a_cerr.astype(int), int(a_cerr.sum())]
    no_can = C.assign(nc=C.forma_canonica.eq(False)).groupby("autor").nc.sum().reindex(ORDEN_AUTOR, fill_value=0)
    out["forma no canónica"] = [*no_can.astype(int), int(no_can.sum())]
    return out


TABLA_AUTORES = tabla_autores(CITAS)
lo, hi = comun.intervalo_wilson(int((CITAS.clase == "exacta").sum()), len(CITAS))
display(TABLA_AUTORES.round(2))
print(f"Citas exactas en total: {100 * (CITAS.clase == 'exacta').mean():.2f} % (IC 95 % {100 * lo:.2f}-{100 * hi:.2f} %)")

### 3.2 Incidencias, una por fila

In [ ]:
INCIDENCIAS = CITAS[(CITAS.clase != "exacta") | CITAS.a_cerrada][
    ["imagen", "autor", "bloque", "ruta", "escrito", "real", "clase", "a_cerrada"]]
display(INCIDENCIAS.reset_index(drop=True))

### 3.3 Controles del crítico

- **arbitraje transcrito = recomputado** con `PrioridadEngine` sobre los mismos informes y la misma etiqueta (las listas
  de métricas se comparan como conjuntos; `orden_dimensiones`, en orden);
- **etiqueta del crítico = etiqueta del orquestador**;
- **`afirmaciones` en el orden del arbitraje** (criterio de E3; exacto, porque `afirmaciones` es una lista estructurada);
- **salvedades suficientes:** entre el nº de dimensiones con alguna métrica cerrada y el nº de métricas cerradas (el
  crítico escribe una por gate; las 3 de composición comparten uno);
- **`campos_citados` coherente** con lo citado en el texto (control secundario);
- **cifras sin cita en la síntesis** (heurística: números con decimales o de 2+ dígitos fuera de corchetes).

In [ ]:
CTRL = ["arbitraje_igual_recomputado", "etiqueta_igual_orquestador", "afirmaciones_en_orden",
        "salvedades_suficientes", "campos_citados_coherentes"]
CONTROLES["sintesis_sin_cifras_sueltas"] = CONTROLES.cifras_sin_cita_sintesis.str.len() == 0
CTRL.append("sintesis_sin_cifras_sueltas")
TABLA_CRITICO = pd.DataFrame({
    "cumple": [int(CONTROLES[c].sum()) for c in CTRL],
    "n": len(CONTROLES),
    "%": [100 * CONTROLES[c].mean() for c in CTRL],
    "imágenes que no cumplen": [", ".join(CONTROLES.loc[~CONTROLES[c].astype(bool), "imagen"]) for c in CTRL],
}, index=CTRL)
display(TABLA_CRITICO.round(1))
difs = CONTROLES[~CONTROLES.arbitraje_igual_recomputado][["imagen", "etiqueta_critico", "arbitraje_diferencias"]]
display(difs)
print("Tensiones por crítica:", CONTROLES.n_tensiones.value_counts().sort_index().to_dict())

### 3.4 Fidelidad de la transcripción del informe

El `informe` de cada `salida_*.json` no es la salida directa de la herramienta: lo copia el LLM (experimentos.md §2.1).
Se compara campo a campo con el informe que da la herramienta sobre la misma imagen (caché de gates). `ruido` es una
diferencia relativa < 1e-6 (variación numérica de la herramienta, no del LLM).

In [ ]:
def tabla_fidelidad(F):
    t = pd.crosstab(F.dimension, F.estado).reindex(index=comun.DIMENSIONES, fill_value=0)
    t.loc["total"] = t.sum()
    t["% igual"] = 100 * t.get("igual", 0) / t.sum(axis=1)
    return t


INF_POR_EJEC = FIDELIDAD.groupby(["imagen", "rep", "dimension"]).estado.apply(lambda e: (e == "igual").all())
TABLA_FIDELIDAD = tabla_fidelidad(FIDELIDAD)
display(TABLA_FIDELIDAD.round(2))
print(f"Informes transcritos idénticos a la herramienta: {INF_POR_EJEC.sum()}/{len(INF_POR_EJEC)} "
      f"({100 * INF_POR_EJEC.mean():.1f} %)")
display(FIDELIDAD[FIDELIDAD.estado != "igual"][["imagen", "dimension", "campo", "estado", "transcrito", "herramienta"]])

### 3.5 Fallos en el primer intento

Ejecuciones que no produjeron las 6 salidas (el lanzador las repite; el intento fallido queda archivado). El motivo es la
última línea de validación del traceback.

In [ ]:
def motivo(error):
    lineas = [l.strip() for l in (error or "").splitlines() if l.strip()]
    val = [l for l in lineas if "validation error" in l]
    detalle = next((lineas[i + 1] + " · " + lineas[i + 2] for i, l in enumerate(lineas)
                    if "validation error" in l and i + 2 < len(lineas)), None)
    return (val[-1] + " · " + detalle) if val and detalle else (lineas[-1] if lineas else "")


FALLOS = pd.DataFrame([{"tanda": r["tanda"], "rep": r["rep"], "imagen": r["imagen"], "intento": r["intento"],
                        "salidas_que_faltan": ", ".join(r["salidas_que_faltan"]), "motivo": motivo(r["error"]),
                        "coste_usd": r["coste_usd_estimado"]}
                       for t in REG for r in REG[t] if r["estado"] != "ok"])
display(FALLOS)
n1 = len({r["imagen"] for r in REG["T100"]})
print(f"T100: {len(FALLOS[FALLOS.tanda == 'T100'])}/{n1} imágenes fallaron en el primer intento; "
      f"tras reintentar, {len(T100)}/{n1} completas")

### 3.6 Criterio de E3

In [ ]:
def resultado(valor, umbral, sentido):
    if umbral is None:
        return "pendiente"
    return "sí" if (valor <= umbral if sentido == "<=" else valor >= umbral) else "no"


n_cerr = int(CITAS.a_cerrada.sum())
pct_exactas = 100 * (CITAS.clase == "exacta").mean()
pct_orden = 100 * CONTROLES.afirmaciones_en_orden.mean()
CRITERIO_E3 = pd.DataFrame([
    ("Citas a métricas cerradas", "<=", CRIT_E3["citas_a_cerradas_max"], n_cerr),
    ("% de citas exactas", ">=", CRIT_E3["pct_citas_exactas_min"], round(pct_exactas, 2)),
    ("% de críticas con afirmaciones en orden", ">=", CRIT_E3["pct_criticas_en_orden_min"], round(pct_orden, 2)),
], columns=["criterio", "sentido", "umbral", "valor"])
CRITERIO_E3["resultado"] = [resultado(v, u, s) for v, u, s in CRITERIO_E3[["valor", "umbral", "sentido"]].values]
GLOBAL_E3 = "sí" if (CRITERIO_E3.resultado == "sí").all() else "no" if (CRITERIO_E3.resultado == "no").any() else "pendiente"
display(Markdown(f"**Criterio global de E3: {GLOBAL_E3}**"))
CRITERIO_E3

## 4. E4b · Coste y latencia (T100)

- **Latencia:** tiempo de pared del `kickoff` completo. La primera ejecución del proceso incluye la carga de YOLO y la
  ResNet y se excluye de la mediana y del p95. Por fases: orquestador (fin de su tarea), Nivel 2 (hasta que acaba el
  último especialista; van en paralelo) y crítico (resto).
- **Tokens:** por instancia de LLM (CrewAI) y de la lectura visual (llamada directa a `google-genai`). La salida incluye
  los tokens de razonamiento, que se cobran como salida.
- **Coste:** tokens × precios de `config_experimentos.yaml` (consulta del 2026-09-28). Es una **cota superior**: no
  descuenta la caché implícita de Gemini. Se contrasta con el panel de gasto de AI Studio.

In [ ]:
def rol(agentes):
    t = " ".join(agentes).lower()
    for clave, nombre in [("orquestador", "orquestador"), ("crítico", "crítico"), ("composición", "composición"),
                          ("líneas", "líneas"), ("luz", "luz y tono"), ("espacio", "espacio")]:
        if clave in t:
            return nombre
    return "otro"


def fila_coste(reg):
    precios = PRECIOS["modelos"]
    f = {"imagen": reg["imagen"], "tanda": reg["tanda"], "rep": reg["rep"], "estado": reg["estado"],
         "archivado": reg["archivado"], "segundos": reg["segundos"], "primera": reg["primera_del_proceso"],
         "vram_mb": reg["vram_pico_mb"], "coste_usd": reg["coste_usd_estimado"]}
    for u in reg["tokens_llm"]:
        r = rol(u["agentes"])
        p = precios[u["modelo"].split("/")[-1]]
        f[f"in_{r}"], f[f"out_{r}"] = u["prompt_tokens"], u["completion_tokens"]
        f[f"razon_{r}"] = u["reasoning_tokens"]
        f[f"usd_{r}"] = (u["prompt_tokens"] * p["entrada"] + u["completion_tokens"] * p["salida"]) / 1e6
    lv = reg["tokens_lectura_visual"]
    if lv:
        p = precios[lv["modelo"]]
        f["in_lectura"], f["out_lectura"] = lv["prompt_tokens"], lv["candidates_tokens"] + lv["thoughts_tokens"]
        f["usd_lectura"] = (f["in_lectura"] * p["entrada"] + f["out_lectura"] * p["salida"]) / 1e6
    marcas = {t["tarea"]: t["t_fin_s"] for t in reg["tareas"]}
    nivel2 = [v for k, v in marcas.items() if k not in ("compose_task", "critica_task")]
    fin_n2 = max(nivel2) if len(nivel2) == 4 else np.nan   # un intento fallido puede no tener las marcas
    f["t_orquestador"] = marcas.get("compose_task", np.nan)
    f["t_nivel2"] = fin_n2 - f["t_orquestador"]
    f["t_critico"] = marcas.get("critica_task", np.nan) - fin_n2
    return f


COSTE = pd.DataFrame([fila_coste(r) for r in REG["T100"]])
COSTE[[c for c in COSTE if c.split("_")[0] in ("in", "out", "razon", "usd")]] = COSTE[
    [c for c in COSTE if c.split("_")[0] in ("in", "out", "razon", "usd")]].fillna(0)
OK = COSTE[~COSTE.archivado]
CALIENTE = OK[~OK.primera]
ROLES = ["orquestador", "composición", "líneas", "luz y tono", "espacio", "crítico", "lectura"]
USD_POR_EUR = PRECIOS["usd_por_eur"]


def resumen(serie):
    return pd.Series({"media": serie.mean(), "mediana": serie.median(), "p95": serie.quantile(0.95),
                      "mín": serie.min(), "máx": serie.max()})


LATENCIA = pd.DataFrame({"total (s)": resumen(CALIENTE.segundos), "orquestador (s)": resumen(CALIENTE.t_orquestador),
                         "nivel 2 (s)": resumen(CALIENTE.t_nivel2), "crítico (s)": resumen(CALIENTE.t_critico)}).T
display(LATENCIA.round(1))

TOKENS = pd.DataFrame({
    "entrada (media)": [OK[f"in_{r}"].mean() for r in ROLES],
    "salida con razonamiento (media)": [OK[f"out_{r}"].mean() for r in ROLES],
    "de ella, razonamiento": [OK.get(f"razon_{r}", pd.Series(0, index=OK.index)).mean() for r in ROLES],
    "USD por imagen (media)": [OK[f"usd_{r}"].mean() for r in ROLES],
}, index=ROLES)
TOKENS["% del coste"] = 100 * TOKENS["USD por imagen (media)"] / TOKENS["USD por imagen (media)"].sum()
TOKENS.loc["total"] = TOKENS.sum()
display(TOKENS.round(4))

GASTO_TOTAL_USD = COSTE.coste_usd.sum()
COSTE_RESUMEN = {
    "usd_por_imagen_media": OK.coste_usd.mean(), "usd_por_imagen_mediana": OK.coste_usd.median(),
    "usd_por_imagen_p95": OK.coste_usd.quantile(0.95),
    "eur_por_imagen_media": OK.coste_usd.mean() / USD_POR_EUR,
    "gasto_T100_usd_incluidos_fallidos": GASTO_TOTAL_USD,
    "gasto_fallidos_usd": COSTE[COSTE.archivado].coste_usd.sum(),
    "vram_pico_mb_max": OK.vram_mb.max(), "vram_pico_mb_mediana": OK.vram_mb.median(),
    "fallos_primer_intento": int(COSTE.archivado.sum()), "n_imagenes": len(OK),
}
print({k: round(v, 4) if isinstance(v, float) else v for k, v in COSTE_RESUMEN.items()})

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axs[0]
ax.hist(CALIENTE.segundos, bins=20, color=AZUL, alpha=0.85, edgecolor="white")
for q, estilo in [(CALIENTE.segundos.median(), "-"), (CALIENTE.segundos.quantile(0.95), "--")]:
    ax.axvline(q, color=NARANJA, ls=estilo, lw=1.5)
ax.set(xlabel="segundos por análisis", ylabel="imágenes")
ax.set_title(f"Latencia en T100 (n = {len(CALIENTE)}; mediana y p95)", loc="left")
ax.grid(axis="x", visible=False)

ax = axs[1]
usd = TOKENS.loc[ROLES, "USD por imagen (media)"]
ax.barh(ROLES[::-1], usd[::-1], color=[NARANJA if r == "crítico" else AZUL for r in ROLES[::-1]])
for y, v in enumerate(usd[::-1]):
    ax.text(v + usd.max() * 0.02, y, f"{v:.4f}", va="center", fontsize=8, color=TINTA2)
ax.set(xlabel="USD por imagen (media, cota superior)")
ax.set_title("Coste por agente", loc="left")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(RES4 / "latencia_coste_T100.png", bbox_inches="tight")
plt.show()

## 5. E4c · Reproducibilidad (R: 5 imágenes × 3 repeticiones)

La lectura visual se reutiliza de la caché (decisión §15.2-3), así que la variabilidad medida es la de los especialistas
y del crítico. Por imagen:

- **informes transcritos idénticos a la herramienta** en las 3 repeticiones (4 informes × 3);
- **arbitraje idéntico** entre las 3 repeticiones (y etiqueta del orquestador idéntica);
- **citas repetidas:** índice de Jaccard medio entre pares de repeticiones del conjunto de citas `(campo, valor escrito)`,
  por autor; y nº de citas por ejecución.

In [ ]:
CITAS_R, CONTROLES_R, FIDELIDAD_R, ARB_R = auditar_tanda(R)


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0


filas = []
for s in LISTA_R:
    arbs = [ARB_R[(s, k)] for k in (1, 2, 3)]
    etiquetas = CONTROLES_R[CONTROLES_R.imagen == s].etiqueta_orquestador.tolist()
    inf = FIDELIDAD_R[FIDELIDAD_R.imagen == s].groupby(["rep", "dimension"]).estado.apply(lambda e: (e == "igual").all())
    fila = {"imagen": s, "contexto": etiquetas[0],
            "informes idénticos a la herramienta": f"{int(inf.sum())}/{len(inf)}",
            "arbitraje idéntico en las 3": all(a == arbs[0] for a in arbs),
            "etiqueta idéntica en las 3": len(set(etiquetas)) == 1,
            "citas por ejecución": round(len(CITAS_R[CITAS_R.imagen == s]) / 3, 1),
            "% exactas": round(100 * (CITAS_R[CITAS_R.imagen == s].clase == "exacta").mean(), 1)}
    for autor in ORDEN_AUTOR:
        conj = [set(map(tuple, CITAS_R[(CITAS_R.imagen == s) & (CITAS_R.rep == k) & (CITAS_R.autor == autor)]
                        [["ruta", "escrito"]].values)) for k in (1, 2, 3)]
        fila[f"Jaccard {autor}"] = round(np.mean([jaccard(conj[i], conj[j]) for i, j in ((0, 1), (0, 2), (1, 2))]), 3)
    filas.append(fila)
REPRO = pd.DataFrame(filas).set_index("imagen")
display(REPRO)

INF_R = FIDELIDAD_R.groupby(["imagen", "rep", "dimension"]).estado.apply(lambda e: (e == "igual").all())
PCT_INF_IDENTICOS = 100 * INF_R.mean()
PCT_ARB_IDENTICO = 100 * REPRO["arbitraje idéntico en las 3"].mean()
JACCARD_MEDIO = REPRO[[c for c in REPRO if c.startswith("Jaccard")]].mean()
print(f"Informes idénticos a la herramienta en R: {int(INF_R.sum())}/{len(INF_R)} ({PCT_INF_IDENTICOS:.1f} %) · "
      f"arbitraje idéntico entre repeticiones: {PCT_ARB_IDENTICO:.0f} % de las imágenes")
print("Jaccard medio de las citas entre repeticiones:", JACCARD_MEDIO.round(3).to_dict())
display(FIDELIDAD_R[FIDELIDAD_R.estado != "igual"][["imagen", "rep", "dimension", "campo", "estado", "transcrito", "herramienta"]])

### 5.1 Criterio de E4

In [ ]:
CRITERIO_E4 = pd.DataFrame([
    ("% de informes transcritos idénticos a la herramienta (R)", ">=", CRIT_E4["pct_informes_identicos_min"],
     round(PCT_INF_IDENTICOS, 2)),
    ("% de imágenes de R con arbitraje idéntico en las 3 repeticiones", ">=", CRIT_E4["pct_informes_identicos_min"],
     round(PCT_ARB_IDENTICO, 2)),
], columns=["criterio", "sentido", "umbral", "valor"])
CRITERIO_E4["resultado"] = [resultado(v, u, s) for v, u, s in CRITERIO_E4[["valor", "umbral", "sentido"]].values]
GLOBAL_E4 = "sí" if (CRITERIO_E4.resultado == "sí").all() else "no" if (CRITERIO_E4.resultado == "no").any() else "pendiente"
display(Markdown(f"**Criterio global de E4: {GLOBAL_E4}** (el coste y la latencia son descriptivos: no tienen umbral)"))
CRITERIO_E4

## 6. Exportación

- `resultados/E3/`: `citas_T100.csv` (una fila por cita), `controles_critico_T100.csv`, `fidelidad_T100.csv`,
  `tabla_por_autor.csv`, `tabla_critico.csv`, `tabla_fidelidad.csv`, `incidencias.csv`, `fallos.csv`, `metricas.json`;
- `resultados/E4/`: `registro_T100.csv` (tiempos, tokens y coste por ejecución), `latencia.csv`, `tokens_coste.csv`,
  `reproducibilidad_R.csv`, `citas_R.csv`, `metricas.json` y la figura.

In [ ]:
CITAS.to_csv(RES3 / "citas_T100.csv", index=False, encoding="utf-8")
CONTROLES.to_csv(RES3 / "controles_critico_T100.csv", index=False, encoding="utf-8")
FIDELIDAD.to_csv(RES3 / "fidelidad_T100.csv", index=False, encoding="utf-8")
TABLA_AUTORES.to_csv(RES3 / "tabla_por_autor.csv", encoding="utf-8")
TABLA_CRITICO.to_csv(RES3 / "tabla_critico.csv", encoding="utf-8")
TABLA_FIDELIDAD.to_csv(RES3 / "tabla_fidelidad.csv", encoding="utf-8")
INCIDENCIAS.to_csv(RES3 / "incidencias.csv", index=False, encoding="utf-8")
FALLOS.to_csv(RES3 / "fallos.csv", index=False, encoding="utf-8")
comun.guardar_json({
    "experimento": "E3", "procedencia": PROC, "congelado": CONGELADO, "commit_ejecuciones": CONGELACION["commit"],
    "n_ejecuciones": len(CONTROLES), "n_citas": len(CITAS),
    "tabla_por_autor": TABLA_AUTORES.reset_index().to_dict("records"),
    "tabla_critico": TABLA_CRITICO.reset_index().to_dict("records"),
    "fidelidad": {"informes_identicos": int(INF_POR_EJEC.sum()), "informes": len(INF_POR_EJEC),
                  "campos_distintos": FIDELIDAD[FIDELIDAD.estado != "igual"].to_dict("records")},
    "fallos": FALLOS.to_dict("records"),
    "criterio": {"por_criterio": CRITERIO_E3.to_dict("records"), "global": GLOBAL_E3},
}, RES3 / "metricas.json")

COSTE.to_csv(RES4 / "registro_T100.csv", index=False, encoding="utf-8")
LATENCIA.to_csv(RES4 / "latencia.csv", encoding="utf-8")
TOKENS.to_csv(RES4 / "tokens_coste.csv", encoding="utf-8")
REPRO.to_csv(RES4 / "reproducibilidad_R.csv", encoding="utf-8")
CITAS_R.to_csv(RES4 / "citas_R.csv", index=False, encoding="utf-8")
comun.guardar_json({
    "experimento": "E4", "procedencia": PROC, "congelado": CONGELADO, "commit_ejecuciones": CONGELACION["commit"],
    "precios": PRECIOS, "latencia": LATENCIA.reset_index().to_dict("records"),
    "tokens_coste": TOKENS.reset_index().to_dict("records"), "coste": COSTE_RESUMEN,
    "reproducibilidad": {"lista_R": LISTA_R, "pct_informes_identicos": PCT_INF_IDENTICOS,
                         "pct_arbitraje_identico": PCT_ARB_IDENTICO, "jaccard_medio": JACCARD_MEDIO.to_dict(),
                         "por_imagen": REPRO.reset_index().to_dict("records")},
    "criterio": {"por_criterio": CRITERIO_E4.to_dict("records"), "global": GLOBAL_E4},
}, RES4 / "metricas.json")
sorted(p.name for p in RES3.iterdir()), sorted(p.name for p in RES4.iterdir())

## 7. Limitaciones propias de E3 y E4 (para §8.6 y §9.3-§9.4)

- **El verificador comprueba cifras, no adjetivos:** «clave baja extrema» con una cifra bien citada pasa el control.
  Tampoco ve las cifras escritas sin corchetes (salvo la heurística de la síntesis).
- **Resolución permisiva de las citas al arbitraje:** las formas libres (`lineas_direccion.W`, `W_luz`…) se resuelven y
  se cuentan aparte; con una gramática estricta contarían como campos inexistentes.
- **Fidelidad frente a la caché de gates:** la herramienta se ejecutó en otro proceso con el mismo código congelado. Una
  diferencia minúscula (`ruido`) se atribuye a la herramienta y no al LLM.
- **Sin V20:** no se mide el efecto de la lectura visual sobre las citas del crítico.
- **R tiene 5 imágenes:** la reproducibilidad es indicativa. La lectura visual sale de caché en R.
- **Coste como cota superior:** no descuenta la caché implícita de Gemini; los precios son los de la fecha de consulta.
- **Latencia por fases aproximada:** el Nivel 2 va en paralelo y solo se conoce el fin de cada tarea.